# Visual Analysis and Image Annotation

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook analyzes and annotates images using Google's Gemini multimodal AI. Upload field photos, archival images, or visual research materials and get structured descriptions, tags, categorizations, or custom analyses for each image. Results are exported as structured data for integration with other research tools.

Gemini processes images natively — it sees the visual content directly, not metadata or filenames. This enables detailed ethnographic description, systematic tagging of objects and activities, and consistent categorization across large photo collections.

## Key Features

1. **Multimodal AI Analysis**: Gemini sees and interprets image content directly
2. **4 Analysis Modes**: Describe, Tag, Categorize, or Custom prompt
3. **Research Presets**: Built-in prompt sets for ethnographic fieldwork, material culture, built environment, food and markets, ritual and ceremony
4. **Multi-File Batch**: Upload and analyze multiple images in one session
5. **Structured Output**: Tags, categories, and descriptions returned as structured data
6. **4 Export Formats**: CSV, Excel, JSON, and Word with image references
7. **Pipeline Ready**: Output pairs with the Multimodal Embedding Explorer for cross-modal analysis

## Workflow

1. **Setup**: Install packages and configure your Gemini API key
2. **Configure**: Select analysis mode, research preset or custom prompt
3. **Upload**: Load image files
4. **Analyze**: Generate annotations for each image via the Gemini API
5. **Review**: Examine images with their annotations side by side
6. **Export**: Download structured results

## Applications

This notebook supports any research that involves systematic analysis of visual materials — cataloging field photos, annotating archival images, describing material culture, documenting built environments, or preparing visual data for coding and thematic analysis. The structured output integrates with other toolkit notebooks for cross-modal comparison.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using multimodal AI to assist with the systematic description and categorization of visual research materials. AI-generated descriptions are starting points for analysis, not authoritative interpretations. The researcher's ethnographic knowledge and contextual understanding remain essential.

**Important**: AI image analysis reflects training data biases. Review all annotations critically, especially for culturally specific content, non-Western contexts, and sensitive materials.

## Target Audience

Designed for anthropologists and qualitative researchers who work with visual data — from graduate students organizing fieldwork photos to applied researchers analyzing visual materials across projects.

## Technical Approach

The notebook uses **Gemini 2.5 Flash** via the Google GenAI SDK for multimodal image analysis. Each image is sent to the API with a structured prompt tailored to the selected analysis mode. Responses are parsed into structured fields (description, tags, categories) for export. No local GPU required.

## AI Anthropology Toolkit

This notebook is part of the [AI Anthropology Toolkit](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit), a collection of computational notebooks for AI-assisted qualitative research. It pairs with the [Multimodal Embedding Explorer](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Multimodal_Embedding_Explorer.ipynb) for organizing and mapping mixed-media fieldwork collections.

## License & Attribution

This notebook is released under the [PolyForm Noncommercial License 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0): free for noncommercial use — research, education, and nonprofit work — with attribution to Matt Artz appreciated. For commercial licensing, contact [Matt Artz](https://www.mattartz.me/).

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Anthropological Forum. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install the Google GenAI SDK and supporting libraries. You will need a Gemini API key — get one free at [Google AI Studio](https://aistudio.google.com/apikey).

In [ ]:
# Install required packages
!pip install -q google-genai pandas "numpy<2.2" openpyxl python-docx ipywidgets Pillow

import re
import io
import json
import base64
import unicodedata
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, HTML, clear_output, Image as IPImage
import ipywidgets as widgets
from PIL import Image as PILImage

# Google GenAI
from google import genai
from google.genai import types
print("\u2713 google-genai loaded")

# python-docx
import docx
from docx.shared import Inches, Pt, RGBColor
print("\u2713 python-docx loaded")

# Colab detection
IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    pass

# Utilities
def make_slug(text):
    """Filesystem-safe slug from text."""
    text = unicodedata.normalize('NFKD', text)
    text = re.sub(r'[^\w\s-]', '', text).strip().lower()
    return re.sub(r'[-\s]+', '_', text)[:50] or 'project'

def normalize_text(text):
    """Normalize unicode artifacts in exported text."""
    if not isinstance(text, str):
        return text
    replacements = {
        '\u2011': '-', '\u2010': '-', '\u2012': '-', '\u2013': '-', '\u2014': '-',
        '\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"',
        '\u2026': '...', '\xa0': ' ',
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text

def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    hf = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    hfont = Font(bold=True, color='FFFFFF')
    tb = Border(
        left=Side(style='thin', color='E7ECEF'), right=Side(style='thin', color='E7ECEF'),
        top=Side(style='thin', color='E7ECEF'), bottom=Side(style='thin', color='E7ECEF')
    )
    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.fill = hf
            cell.font = hfont
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = tb
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
                cell.border = tb
    wb.save(filepath)

print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print("\U0001f4f7 Ready to configure image analysis!")

## Configuration

Set your Gemini API key, select an analysis mode, and choose a research preset or write a custom prompt. The analysis mode determines what kind of annotation each image receives.

In [ ]:
# Configuration

ANALYSIS_PROMPTS = {
    'describe': {
        'label': 'Describe',
        'system': (
            'You are an ethnographic research assistant analyzing field photographs. '
            'Provide a detailed, objective description of the image. Include:\n'
            '1. **Setting**: Where this appears to take place (indoor/outdoor, urban/rural, type of space)\n'
            '2. **People**: Number of people, apparent activities, postures, interactions\n'
            '3. **Objects**: Notable objects, tools, materials, technologies present\n'
            '4. **Activities**: What appears to be happening\n'
            '5. **Atmosphere**: Lighting, time of day, weather if visible, overall mood\n'
            '6. **Cultural markers**: Clothing, signage, architectural features, decorations\n\n'
            'Be descriptive and specific. Avoid interpretive claims about meaning or intent. '
            'Note what you observe, not what you assume.'
        ),
    },
    'tag': {
        'label': 'Tag',
        'system': (
            'You are an ethnographic research assistant tagging field photographs. '
            'Extract structured tags from the image. Return ONLY a JSON object with these keys:\n'
            '- "people": list of descriptors (e.g., ["adult male", "child", "group of 3"])\n'
            '- "objects": list of notable objects visible\n'
            '- "activities": list of activities or actions occurring\n'
            '- "setting": list of setting descriptors (e.g., ["outdoor", "market", "urban"])\n'
            '- "materials": list of materials or textures visible\n'
            '- "colors": list of dominant colors\n'
            '- "mood": list of atmospheric descriptors\n\n'
            'Return valid JSON only. Use lowercase tags. Be specific rather than generic.'
        ),
    },
    'categorize': {
        'label': 'Categorize',
        'system': None,  # Set dynamically based on preset
    },
    'custom': {
        'label': 'Custom Prompt',
        'system': None,  # Set by user
    },
}

RESEARCH_PRESETS = {
    'Ethnographic Fieldwork': [
        'Daily life / routine', 'Social gathering', 'Work / labor', 'Market / commerce',
        'Domestic space', 'Public space', 'Ceremony / ritual', 'Food preparation',
        'Transportation', 'Education', 'Healthcare', 'Recreation', 'Portrait',
        'Landscape / environment', 'Infrastructure', 'Other',
    ],
    'Material Culture': [
        'Tool / implement', 'Clothing / textile', 'Container / vessel', 'Furniture',
        'Decoration / ornament', 'Musical instrument', 'Weapon', 'Religious object',
        'Technology / device', 'Food / drink', 'Building material', 'Craft product',
        'Written document', 'Art / sculpture', 'Other',
    ],
    'Built Environment': [
        'Residential', 'Commercial', 'Religious', 'Government / institutional',
        'Industrial', 'Agricultural', 'Infrastructure', 'Public space',
        'Monument / memorial', 'Ruin / abandoned', 'Construction', 'Interior',
        'Streetscape', 'Landscape', 'Other',
    ],
    'Food & Markets': [
        'Market stall / vendor', 'Street food', 'Restaurant / cafe', 'Food preparation',
        'Raw ingredients', 'Cooked dish', 'Beverage', 'Food storage',
        'Agricultural product', 'Fishing / hunting', 'Food packaging',
        'Eating / dining', 'Food ritual', 'Other',
    ],
    'Ritual & Ceremony': [
        'Religious service', 'Life cycle ritual', 'Festival / celebration', 'Funeral / mourning',
        'Healing ritual', 'Offering / sacrifice', 'Procession', 'Dance / performance',
        'Prayer / meditation', 'Sacred space', 'Ritual object', 'Costume / regalia',
        'Music / chanting', 'Other',
    ],
}

class AnalysisConfig:
    """Configuration for image analysis."""
    API_KEY = ''
    MODEL = 'gemini-2.5-flash'
    MODE = 'describe'
    PRESET = 'Ethnographic Fieldwork'
    CUSTOM_PROMPT = ''
    CUSTOM_CATEGORIES = ''
    PROJECT_NAME = ''
    PROJECT_DESCRIPTION = ''

config = AnalysisConfig()
client = None

# Try Colab secrets first
try:
    from google.colab import userdata
    config.API_KEY = userdata.get('GEMINI_API_KEY')
    print("\u2713 API key loaded from Colab Secrets")
except Exception:
    pass

style = {'description_width': '160px'}
layout = widgets.Layout(width='500px')

def create_config_interface():
    global client

    config_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
    config_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f4f7 Visual Analysis</h2>'
    config_html += '<p><strong>Welcome!</strong> This notebook analyzes and annotates images using Gemini\'s multimodal AI.</p>'
    config_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
    config_html += '<ol>'
    config_html += '<li><strong>Configure:</strong> Set your API key and analysis mode below</li>'
    config_html += '<li><strong>Upload:</strong> Load image files</li>'
    config_html += '<li><strong>Analyze:</strong> Generate annotations via the Gemini API</li>'
    config_html += '<li><strong>Review:</strong> Examine images with annotations</li>'
    config_html += '<li><strong>Export:</strong> Download structured results</li>'
    config_html += '</ol>'
    config_html += '</div>'

    instructions = widgets.HTML(value=config_html)

    # API key
    api_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f511 API Key</h3>')

    api_key_input = widgets.Password(
        value=config.API_KEY,
        placeholder='Enter your Gemini API key',
        description='Gemini API Key:',
        layout=layout, style=style
    )

    api_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Get a free API key at <a href="https://aistudio.google.com/apikey" target="_blank">Google AI Studio</a>.</p>'
    )

    # Model
    model_input = widgets.Dropdown(
        options=[
            ('Gemini 2.5 Flash (recommended)', 'gemini-2.5-flash'),
            ('Gemini 2.5 Pro (most capable)', 'gemini-2.5-pro'),
            ('Gemini 2.0 Flash', 'gemini-2.0-flash'),
        ],
        value='gemini-2.5-flash',
        description='Model:',
        layout=layout, style=style
    )

    # Analysis mode
    mode_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Analysis Mode</h3>')

    mode_input = widgets.Dropdown(
        options=[
            ('Describe \u2014 detailed ethnographic description', 'describe'),
            ('Tag \u2014 extract structured tags (JSON)', 'tag'),
            ('Categorize \u2014 classify into categories', 'categorize'),
            ('Custom \u2014 your own analysis prompt', 'custom'),
        ],
        value='describe',
        description='Mode:',
        layout=layout, style=style
    )

    mode_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        '<strong>Describe:</strong> Detailed narrative description of the image.<br>'
        '<strong>Tag:</strong> Structured JSON tags (people, objects, activities, setting).<br>'
        '<strong>Categorize:</strong> Classify images into preset or custom categories.<br>'
        '<strong>Custom:</strong> Write your own analysis prompt.</p>'
    )

    # Categorize preset
    preset_input = widgets.Dropdown(
        options=list(RESEARCH_PRESETS.keys()) + ['Custom Categories'],
        value='Ethnographic Fieldwork',
        description='Preset:',
        layout=layout, style=style
    )

    custom_categories = widgets.Textarea(
        value='',
        placeholder='Enter categories, one per line',
        description='Categories:',
        layout=widgets.Layout(width='500px', height='100px'),
        style=style
    )

    preset_box = widgets.VBox([preset_input, custom_categories])
    preset_box.layout.display = 'none'
    custom_categories.layout.display = 'none'

    # Custom prompt
    custom_prompt = widgets.Textarea(
        value='',
        placeholder='Write your analysis prompt here. The image will be sent along with this prompt.',
        description='Prompt:',
        layout=widgets.Layout(width='500px', height='120px'),
        style=style
    )
    custom_box = widgets.VBox([custom_prompt])
    custom_box.layout.display = 'none'

    def on_mode_change(change):
        preset_box.layout.display = '' if change['new'] == 'categorize' else 'none'
        custom_box.layout.display = '' if change['new'] == 'custom' else 'none'
    mode_input.observe(on_mode_change, names='value')

    def on_preset_change(change):
        custom_categories.layout.display = '' if change['new'] == 'Custom Categories' else 'none'
    preset_input.observe(on_preset_change, names='value')

    # Project metadata
    project_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Project Metadata</h3>')

    project_name = widgets.Text(
        value='', placeholder='e.g., Fieldwork Photos 2026',
        description='Project name:', layout=layout, style=style
    )

    project_desc = widgets.Textarea(
        value='', placeholder='Brief description of the images and research context',
        description='Description:',
        layout=widgets.Layout(width='500px', height='60px'), style=style
    )

    save_btn = widgets.Button(
        description='Save Configuration',
        style={'button_color': '#6096BA'},
        layout=widgets.Layout(width='200px', margin='20px 0')
    )

    status = widgets.Output()

    def save_config(btn):
        global client
        config.API_KEY = api_key_input.value.strip()
        config.MODEL = model_input.value
        config.MODE = mode_input.value
        config.PRESET = preset_input.value
        config.CUSTOM_PROMPT = custom_prompt.value.strip()
        config.CUSTOM_CATEGORIES = custom_categories.value.strip()
        config.PROJECT_NAME = project_name.value.strip()
        config.PROJECT_DESCRIPTION = project_desc.value.strip()

        with status:
            clear_output()
            if not config.API_KEY:
                print("\u26a0\ufe0f Please enter a Gemini API key.")
                return

            print("Validating API key...")
            try:
                client = genai.Client(api_key=config.API_KEY)
                # Quick validation
                test = client.models.generate_content(
                    model=config.MODEL,
                    contents='Say "ok" in one word.',
                    config=types.GenerateContentConfig(max_output_tokens=10)
                )
                print(f"\u2713 API key validated \u2014 {config.MODEL} is accessible")
                print(f"\u2713 Mode: {ANALYSIS_PROMPTS[config.MODE]['label']}")
                if config.MODE == 'categorize':
                    print(f"\u2713 Preset: {config.PRESET}")
                if config.PROJECT_NAME:
                    print(f"\u2713 Project: {config.PROJECT_NAME}")
                print()
                print("\u2713 Configuration saved! Proceed to Upload.")
            except Exception as e:
                print(f"\u274c API validation failed: {e}")
                client = None

    save_btn.on_click(save_config)

    display(widgets.VBox([
        instructions,
        api_header, api_key_input, api_help,
        model_input,
        mode_header, mode_input, mode_help,
        preset_box, custom_box,
        project_header, project_name, project_desc,
        save_btn, status,
    ]))

create_config_interface()

## Upload Images

Upload field photos or other image files. The notebook accepts .png, .jpg, .jpeg, .webp, .tiff, and .bmp formats. Select multiple files at once or upload in batches.

In [ ]:
# Upload Images

IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.webp', '.tiff', '.bmp'}

MIME_MAP = {
    '.png': 'image/png', '.jpg': 'image/jpeg', '.jpeg': 'image/jpeg',
    '.webp': 'image/webp', '.tiff': 'image/tiff', '.bmp': 'image/bmp',
}

uploaded_images = []  # list of dicts: {filename, raw_bytes, mime_type, width, height, size_kb}

def process_image_uploads(change):
    """Process uploaded image files."""
    upload_out.clear_output()
    new_images = []

    for file_info in uploader.value:
        fname = file_info['name'] if isinstance(file_info, dict) else file_info.name
        content = file_info['content'] if isinstance(file_info, dict) else file_info.content
        raw_bytes = bytes(content)

        ext = '.' + fname.rsplit('.', 1)[-1].lower() if '.' in fname else ''
        if ext not in IMAGE_EXTENSIONS:
            with upload_out:
                print(f"\u26a0\ufe0f Skipping {fname} \u2014 unsupported format")
            continue

        if any(img['filename'] == fname for img in uploaded_images):
            continue

        try:
            img = PILImage.open(io.BytesIO(raw_bytes))
            w, h = img.size
            new_images.append({
                'filename': fname,
                'raw_bytes': raw_bytes,
                'mime_type': MIME_MAP.get(ext, 'image/jpeg'),
                'width': w,
                'height': h,
                'size_kb': round(len(raw_bytes) / 1024, 1),
            })
        except Exception as e:
            with upload_out:
                print(f"\u26a0\ufe0f Could not read {fname}: {e}")

    uploaded_images.extend(new_images)

    with upload_out:
        if new_images:
            print(f"\u2713 Added {len(new_images)} image(s) ({len(uploaded_images)} total)")
            print()

            # Thumbnail grid
            cols = min(4, len(uploaded_images))
            grid_html = '<div style="display: flex; flex-wrap: wrap; gap: 10px;">'
            for img_info in uploaded_images:
                b64 = base64.b64encode(img_info['raw_bytes']).decode()
                grid_html += f'<div style="text-align: center; width: 150px;">'
                grid_html += f'<img src="data:{img_info["mime_type"]};base64,{b64}" style="max-width: 150px; max-height: 120px; border-radius: 5px; border: 1px solid #E7ECEF;">'
                grid_html += f'<p style="font-size: 0.8em; color: #274C77; margin: 4px 0 0 0;">{img_info["filename"][:20]}</p>'
                grid_html += f'<p style="font-size: 0.7em; color: #8B8C89; margin: 0;">{img_info["width"]}x{img_info["height"]} \u2022 {img_info["size_kb"]} KB</p>'
                grid_html += '</div>'
            grid_html += '</div>'
            display(HTML(grid_html))

upload_html = '<div style="background-color: #E7ECEF; padding: 15px; border-radius: 10px; margin: 10px 0; border-left: 5px solid #274C77;">'
upload_html += '<h3 style="color: #274C77; margin-top: 0;">\U0001f4c2 Upload Images</h3>'
upload_html += '<p>Supported formats: .png, .jpg, .jpeg, .webp, .tiff, .bmp</p>'
upload_html += '</div>'

upload_instructions = widgets.HTML(value=upload_html)

uploader = widgets.FileUpload(
    accept='.png,.jpg,.jpeg,.webp,.tiff,.bmp',
    multiple=True,
    description='Select images',
    layout=widgets.Layout(width='300px')
)
uploader.observe(process_image_uploads, names='value')

clear_btn = widgets.Button(
    description='Clear All Images',
    style={'button_color': '#8B8C89'},
    layout=widgets.Layout(width='150px', margin='10px 0')
)

upload_out = widgets.Output()

def on_clear(_):
    uploaded_images.clear()
    upload_out.clear_output()
    with upload_out:
        print("\u2713 All images cleared.")

clear_btn.on_click(on_clear)

display(widgets.VBox([upload_instructions, uploader, clear_btn, upload_out]))

## Analyze Images

Send each image to Gemini with the configured analysis prompt. Results are stored for review and export.

In [ ]:
# Analyze Images

import time

all_results = {}  # filename -> {description, tags, category, raw_response}

def build_prompt():
    """Build the analysis prompt based on configuration."""
    mode = config.MODE

    if mode == 'describe':
        return ANALYSIS_PROMPTS['describe']['system']

    elif mode == 'tag':
        return ANALYSIS_PROMPTS['tag']['system']

    elif mode == 'categorize':
        if config.PRESET == 'Custom Categories' and config.CUSTOM_CATEGORIES:
            categories = [c.strip() for c in config.CUSTOM_CATEGORIES.split('\n') if c.strip()]
        else:
            categories = RESEARCH_PRESETS.get(config.PRESET, RESEARCH_PRESETS['Ethnographic Fieldwork'])

        cat_list = '\n'.join(f'- {c}' for c in categories)
        return (
            'You are an ethnographic research assistant categorizing field photographs. '
            'Classify this image into ONE primary category and up to TWO secondary categories '
            'from the list below. Also provide a one-sentence rationale.\n\n'
            f'Categories:\n{cat_list}\n\n'
            'Return ONLY a JSON object with these keys:\n'
            '- "primary_category": the single best-fitting category\n'
            '- "secondary_categories": list of 0-2 additional relevant categories\n'
            '- "rationale": one sentence explaining the classification\n'
            '- "confidence": "high", "medium", or "low"\n\n'
            'Return valid JSON only.'
        )

    elif mode == 'custom':
        return config.CUSTOM_PROMPT or 'Describe what you see in this image.'

    return 'Describe what you see in this image.'


MAX_IMAGE_MB = 18  # inline request ceiling with margin

def _retry_wait(error_text, attempt):
    """Backoff seconds: honor a server-suggested retry delay when present."""
    m = re.search(r'retry(?:_delay)?\D{0,12}(\d+(?:\.\d+)?)', error_text.lower())
    if m:
        return min(float(m.group(1)) + 1, 60)
    return min(2 * (2 ** attempt), 45)

def analyze_with_retry(img_info, prompt, max_retries=4):
    """Analyze with exponential backoff on rate-limit and transient errors."""
    for attempt in range(max_retries + 1):
        try:
            return analyze_image(img_info, prompt)
        except Exception as e:
            msg = str(e).lower()
            transient = ('429' in msg or 'rate' in msg or '503' in msg
                         or 'unavailable' in msg or 'deadline' in msg)
            if transient and attempt < max_retries:
                time.sleep(_retry_wait(msg, attempt))
                continue
            raise

def analyze_image(img_info, prompt):
    """Send a single image to Gemini for analysis."""
    image_part = types.Part.from_bytes(
        data=img_info['raw_bytes'],
        mime_type=img_info['mime_type']
    )

    response = client.models.generate_content(
        model=config.MODEL,
        contents=[prompt, image_part],
        config=types.GenerateContentConfig(
            temperature=0.3,
            max_output_tokens=2048,
        )
    )

    return response.text


analyze_btn = widgets.Button(
    description='Analyze All Images',
    style={'button_color': '#6096BA'},
    layout=widgets.Layout(width='200px', margin='10px 0')
)

progress_html = widgets.HTML(value='')
analyze_out = widgets.Output()

def on_analyze(_):
    global all_results
    analyze_out.clear_output()
    progress_html.value = ''

    if client is None:
        with analyze_out:
            print("\u26a0\ufe0f No API client \u2014 run the Configuration cell first.")
        return

    if not uploaded_images:
        with analyze_out:
            print("\u26a0\ufe0f No images uploaded \u2014 run the Upload cell first.")
        return

    prompt = build_prompt()
    total = len(uploaded_images)

    with analyze_out:
        print(f"\U0001f4f7 Analyzing {total} image(s) with {config.MODEL}...")
        print(f"   Mode: {ANALYSIS_PROMPTS[config.MODE]['label']}")
        print()

    pending = [im for im in uploaded_images if im['filename'] not in all_results]
    if len(pending) < total:
        with analyze_out:
            print(f"   ({total - len(pending)} already analyzed — analyzing the rest)")
    for idx, img_info in enumerate(pending):
        fname = img_info['filename']
        size_mb = len(img_info['raw_bytes']) / (1024 * 1024)
        if size_mb > MAX_IMAGE_MB:
            with analyze_out:
                print(f"❌ {fname}: {size_mb:.1f} MB exceeds the {MAX_IMAGE_MB} MB "
                      "request ceiling — resize and re-upload")
            continue
        progress_html.value = (
            f'<span style="color: #274C77;">Analyzing {idx + 1}/{len(pending)} \u2014 {fname}</span>'
        )

        try:
            raw_response = analyze_with_retry(img_info, prompt)

            result = {
                'filename': fname,
                'mode': config.MODE,
                'raw_response': normalize_text(raw_response.strip()),
                'width': img_info['width'],
                'height': img_info['height'],
            }

            # Parse structured responses
            if config.MODE == 'tag' or config.MODE == 'categorize':
                try:
                    # Extract JSON from response (handle markdown code blocks)
                    json_str = raw_response.strip()
                    if '```' in json_str:
                        json_str = json_str.split('```')[1]
                        if json_str.startswith('json'):
                            json_str = json_str[4:]
                        json_str = json_str.strip()
                    result['parsed'] = json.loads(json_str)
                except (json.JSONDecodeError, IndexError):
                    result['parsed'] = None

            all_results[fname] = result

            with analyze_out:
                preview = raw_response[:100].replace('\n', ' ')
                print(f"\u2713 {fname} \u2014 {preview}...")

        except Exception as e:
            with analyze_out:
                print(f"\u274c {fname}: {e}")

        # Rate limiting
        if idx < total - 1:
            time.sleep(0.5)

    progress_html.value = ''

    with analyze_out:
        print()
        print(f"\u2713 Analysis complete! {len(all_results)}/{total} images processed.")

analyze_btn.on_click(on_analyze)
display(widgets.VBox([analyze_btn, progress_html, analyze_out]))

## Review Results

Browse images with their annotations side by side. Use the dropdown to switch between images.

In [ ]:
# Review Results

if not all_results:
    print("\u26a0\ufe0f Run the Analyze cell first.")
else:
    file_dropdown = widgets.Dropdown(
        options=list(all_results.keys()),
        description='Image:',
        style={'description_width': '60px'},
        layout=widgets.Layout(width='500px')
    )

    review_out = widgets.Output()

    def show_result(change=None):
        review_out.clear_output()
        fname = file_dropdown.value
        result = all_results[fname]

        # Find matching image data
        img_info = next((img for img in uploaded_images if img['filename'] == fname), None)

        with review_out:
            # Side-by-side layout: image left, annotation right
            b64 = base64.b64encode(img_info['raw_bytes']).decode() if img_info else ''

            review_html = '<div style="display: flex; gap: 20px; align-items: flex-start; flex-wrap: wrap;">'

            # Image
            if img_info:
                review_html += '<div style="flex: 0 0 auto;">'
                review_html += f'<img src="data:{img_info["mime_type"]};base64,{b64}" '
                review_html += 'style="max-width: 400px; max-height: 400px; border-radius: 8px; border: 2px solid #E7ECEF;">'
                review_html += f'<p style="font-size: 0.8em; color: #8B8C89; margin: 5px 0;">{fname} \u2022 {img_info["width"]}x{img_info["height"]}</p>'
                review_html += '</div>'

            # Annotation
            review_html += '<div style="flex: 1; min-width: 300px;">'
            review_html += f'<div style="background-color: #E7ECEF; padding: 15px; border-radius: 10px; border-left: 5px solid #274C77;">'
            review_html += f'<h3 style="color: #274C77; margin-top: 0;">{ANALYSIS_PROMPTS[result["mode"]]["label"]} Analysis</h3>'

            response = result['raw_response']

            if result['mode'] == 'tag' and result.get('parsed'):
                # Format tags nicely
                parsed = result['parsed']
                for key, values in parsed.items():
                    if isinstance(values, list) and values:
                        tags_html = ', '.join(f'<span style="background: #A3CEF1; padding: 2px 8px; border-radius: 12px; font-size: 0.85em;">{v}</span>' for v in values)
                        review_html += f'<p><strong style="color: #274C77;">{key}:</strong> {tags_html}</p>'
                    elif isinstance(values, str):
                        review_html += f'<p><strong style="color: #274C77;">{key}:</strong> {values}</p>'

            elif result['mode'] == 'categorize' and result.get('parsed'):
                parsed = result['parsed']
                review_html += f'<p><strong style="color: #274C77;">Primary:</strong> <span style="background: #6096BA; color: white; padding: 3px 10px; border-radius: 12px;">{parsed.get("primary_category", "N/A")}</span></p>'
                secondary = parsed.get('secondary_categories', [])
                if secondary:
                    sec_html = ', '.join(f'<span style="background: #A3CEF1; padding: 2px 8px; border-radius: 12px; font-size: 0.85em;">{s}</span>' for s in secondary)
                    review_html += f'<p><strong style="color: #274C77;">Secondary:</strong> {sec_html}</p>'
                rationale = parsed.get('rationale', '')
                if rationale:
                    review_html += f'<p style="color: #8B8C89; font-style: italic;">{rationale}</p>'
                conf = parsed.get('confidence', '')
                if conf:
                    review_html += f'<p><strong style="color: #274C77;">Confidence:</strong> {conf}</p>'

            else:
                # Plain text (describe or custom)
                formatted = response.replace('\n', '<br>')
                review_html += f'<div style="line-height: 1.6;">{formatted}</div>'

            review_html += '</div></div></div>'
            display(HTML(review_html))

    file_dropdown.observe(show_result, names='value')
    display(file_dropdown)
    display(review_out)
    show_result()

## Export

Export annotations as CSV, Excel, JSON, or Word. Structured data includes filenames, analysis mode, and all generated annotations.

In [ ]:
# Export

if not all_results:
    print("\u26a0\ufe0f Run the Analyze cell first.")
else:
    export_format = widgets.Dropdown(
        options=[
            ('CSV (.csv) \u2014 structured annotations', 'csv'),
            ('Excel (.xlsx) \u2014 styled with metadata', 'excel'),
            ('JSON (.json) \u2014 full results with parsed data', 'json'),
            ('Word (.docx) \u2014 formatted report with image references', 'docx'),
        ],
        value='csv',
        description='Format:',
        style={'description_width': '80px'},
        layout=widgets.Layout(width='500px')
    )

    export_btn = widgets.Button(
        description='Export',
        style={'button_color': '#6096BA'},
        layout=widgets.Layout(width='200px', margin='10px 0')
    )

    export_out = widgets.Output()

    def build_export_df():
        """Build a DataFrame from results."""
        rows = []
        for fname, result in all_results.items():
            row = {
                'filename': fname,
                'mode': result['mode'],
                'width': result.get('width', ''),
                'height': result.get('height', ''),
            }

            if result['mode'] == 'tag' and result.get('parsed'):
                parsed = result['parsed']
                for key, values in parsed.items():
                    if isinstance(values, list):
                        row[f'tag_{key}'] = '; '.join(str(v) for v in values)
                    else:
                        row[f'tag_{key}'] = str(values)
                row['annotation'] = result['raw_response']
            elif result['mode'] == 'categorize' and result.get('parsed'):
                parsed = result['parsed']
                row['primary_category'] = parsed.get('primary_category', '')
                row['secondary_categories'] = '; '.join(parsed.get('secondary_categories', []))
                row['rationale'] = parsed.get('rationale', '')
                row['confidence'] = parsed.get('confidence', '')
                row['annotation'] = result['raw_response']
            else:
                row['annotation'] = result['raw_response']

            rows.append(row)
        return pd.DataFrame(rows)

    def on_export(_):
        export_out.clear_output()
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        slug = make_slug(config.PROJECT_NAME) if config.PROJECT_NAME else 'annotations'
        fmt = export_format.value

        try:
            if fmt == 'csv':
                filepath = f'{slug}_{timestamp}.csv'
                df = build_export_df()
                df.to_csv(filepath, index=False)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB:
                        colab_files.download(filepath)

            elif fmt == 'excel':
                filepath = f'{slug}_{timestamp}.xlsx'
                df = build_export_df()
                with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
                    df.to_excel(writer, sheet_name='Annotations', index=False)
                style_excel(filepath)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB:
                        colab_files.download(filepath)

            elif fmt == 'json':
                filepath = f'{slug}_{timestamp}.json'
                output = {
                    'metadata': {
                        'project_name': config.PROJECT_NAME,
                        'project_description': config.PROJECT_DESCRIPTION,
                        'model': config.MODEL,
                        'mode': config.MODE,
                        'n_images': len(all_results),
                        'exported_at': datetime.now().isoformat(),
                    },
                    'images': [],
                }
                for fname, result in all_results.items():
                    item = {
                        'filename': fname,
                        'mode': result['mode'],
                        'width': result.get('width'),
                        'height': result.get('height'),
                        'annotation': result['raw_response'],
                    }
                    if result.get('parsed'):
                        item['parsed'] = result['parsed']
                    output['images'].append(item)

                with open(filepath, 'w') as f:
                    json.dump(output, f, indent=2, default=str)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB:
                        colab_files.download(filepath)

            elif fmt == 'docx':
                filepath = f'{slug}_{timestamp}.docx'
                doc_obj = docx.Document()
                doc_obj.add_heading('Visual Analysis Report', level=1)

                meta = f"Project: {config.PROJECT_NAME}" if config.PROJECT_NAME else ''
                meta += f" | Mode: {ANALYSIS_PROMPTS[config.MODE]['label']}"
                meta += f" | Model: {config.MODEL}"
                meta += f" | Images: {len(all_results)}"
                p = doc_obj.add_paragraph(meta)
                p.runs[0].font.color.rgb = RGBColor(0x8B, 0x8C, 0x89)

                doc_obj.add_paragraph()

                for fname, result in all_results.items():
                    # Image heading
                    h = doc_obj.add_heading(fname, level=2)
                    h.runs[0].font.color.rgb = RGBColor(0x27, 0x4C, 0x77)

                    dim = f"{result.get('width', '?')}x{result.get('height', '?')}"
                    p = doc_obj.add_paragraph(f"Dimensions: {dim}")
                    p.runs[0].font.color.rgb = RGBColor(0x8B, 0x8C, 0x89)
                    p.runs[0].font.size = Pt(9)

                    # Annotation
                    annotation = result['raw_response']
                    for line in annotation.split('\n'):
                        line = line.strip()
                        if not line:
                            continue
                        p = doc_obj.add_paragraph()
                        if line.startswith('**') and '**' in line[2:]:
                            # Bold header
                            bold_text = line.split('**')[1]
                            rest = line.split('**', 2)[-1]
                            run = p.add_run(bold_text)
                            run.bold = True
                            run.font.color.rgb = RGBColor(0x27, 0x4C, 0x77)
                            if rest:
                                p.add_run(rest)
                        else:
                            p.add_run(line)

                    doc_obj.add_paragraph()  # spacer

                doc_obj.save(filepath)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB:
                        colab_files.download(filepath)

        except Exception as e:
            with export_out:
                print(f"\u274c Export error: {type(e).__name__}: {e}")

    export_btn.on_click(on_export)
    display(widgets.VBox([export_format, export_btn, export_out]))